<a href="https://colab.research.google.com/github/EmIbrahimovic/sp26_6_4110_hw_colabs/blob/main/hw09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Propositional Logic

In [ ]:
!pip install sympy==1.10

## Imports and Utilities
**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [ ]:
import numpy as np
import sympy

## Propositional Sentence Evaluation

### Utilities


**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [ ]:

import collections

## Common logic data structures
Not = collections.namedtuple("Not", ["sentence"])
And = collections.namedtuple("And", ["sentence1", "sentence2"])
Or = collections.namedtuple("Or", ["sentence1", "sentence2"])
Implies = collections.namedtuple("Implies", ["sentence1", "sentence2"])

## Propositional logic data structures
Proposition = str  # Name of the proposition
PropositionalModel = dict  # Proposition -> bool

## Example of PropositionalModel, used in tests
IS_RAINING = Proposition("is-raining")
IS_SUNNY = Proposition("is-sunny")
NEED_UMBRELLA = Proposition("need-umbrella")
PROP_MODEL = PropositionalModel({
    IS_RAINING: True,
    IS_SUNNY: False,
    NEED_UMBRELLA: True,
})

### Question
*Note: for these questions, refer to the top of the Colab notebook.*
Write a function that takes a propositional sentence and evaluates it against a single model.
You may find python's builtin `isinstance` useful. For example, `isinstance(sentence, And)` returns whether a sentence is an `And`.

For reference, our solution is **15** line(s) of code.

In [ ]:
def evaluate_propositional_sentence(sentence, model):
    """Evaluate a propositional sentence against a single model.

    Args:
      sentence: A Proposition, And, Or, Not, or Implies.
      model: A PropositionalModel.

    Returns:
      holds: A bool representing the truth value of the sentence
        under the model.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:
assert evaluate_propositional_sentence(IS_RAINING, PROP_MODEL) == True


assert evaluate_propositional_sentence(Not(IS_RAINING), PROP_MODEL) == False


assert evaluate_propositional_sentence(And(IS_RAINING, IS_SUNNY),
                                       PROP_MODEL) == False


assert evaluate_propositional_sentence(Implies(IS_SUNNY, Not(IS_RAINING)),
                                       PROP_MODEL) == True


assert evaluate_propositional_sentence(Or(IS_RAINING, Not(IS_RAINING)),
                                       PROP_MODEL) == True

assert evaluate_propositional_sentence(And(Or(Not(IS_RAINING), Not(Not(NEED_UMBRELLA))),
                                           Or(IS_SUNNY, Not(Not(Implies(Not(IS_RAINING), IS_SUNNY))))),
                                       PROP_MODEL) == True

print('Tests passed.')

## Warmup


### Utilities


**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [ ]:
def lit_to_var_val(literal):
    """Converts a literal into (variable, value).

    Args:
      literal: A nonzero int.

    Returns:
      variable: A positive int representing a variable.
      value: True or False, i.e., positive or negative.
    """
    return abs(literal), literal > 0


def is_cnf_formula(formula):
    """Checks whether the input is a valid CNF formula.

    A formula is in valid CNF form if it is a list of lists of nonzero
    integers with sign indicating whether the variable is negated.

    You will not need to use this utility in your implementation,
    but it may be useful to read to understand the CNF representation.
    """
    if not isinstance(formula, list):
        return False
    if len(formula) == 0:
        return True
    clause = formula[0]
    if not isinstance(clause, list):
        return False
    for literal in clause:
        if not isinstance(literal, int):
            return False
        if literal == 0:
            return False
    if len(formula) == 1:
        return True
    return is_cnf_formula(formula[1:])


def get_variables_in_cnf_formula(cnf_formula):
    """Get a list of all variables in a CNF formula.

    Args:
      cnf_formula: A list of lists of nonzero ints.

    Returns:
      variables: A list of all variables that appear in
        the formula.
    """
    variables = set()
    for clause in cnf_formula:
        variables.update({lit_to_var_val(literal)[0] for literal in clause})
    variables = sorted(variables)
    return variables

### Question
In this problem, CNF formulas are represented as lists of lists of nonzero integers. The sign of the integer represents whether the corresponding proposition is negated or not. For example, the formula ((x1 or not x2) and (x3 or x2)) would be represented as [[1, -2], [3, 2]]. Complete the following function to confirm your understanding of this representation.

For reference, our solution is **2** line(s) of code.

In [ ]:
def warmup():
    """Return a list of lists of ints for the CNF formula:

    ((x4 or not x5 or not x6) and (x6 or x5 or not x1) and (x2 or x3)).

    Keep the same order as in the formula above.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:
# Note: the catsoop test is more strict for this question
# than the unit tests here. Make sure that your answer
# matches the description in the docstring exactly.
assert is_cnf_formula(warmup())
assert get_variables_in_cnf_formula(warmup()) == [1, 2, 3, 4, 5, 6]

print('Tests passed.')

## [OPTIONAL] DPLL


### Utilities

**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [ ]:


def clause_is_determined(clause, partial_assignment):
    """Checks whether all variables in the clause have an assignment.

    Args:
      clause: A list of nonzero ints.
      partial_assignment: A dict of variables (ints) to values (bools).

    Returns:
      is_determined: True if all variables in the clause appear in
        partial_assignment.
    """
    for literal in clause:
        if not (literal in partial_assignment or
                -literal in partial_assignment):
            return False
    return True


def literal_is_satisfied(literal, partial_assignment):
    """Checks whether the literal is satisfied by the assignment.

    Args:
      literal: A nonzero int.
      partial_assignment: A dict of variables (ints) to values (bools).

    Returns:
      is_satisfied: True if the literal's variable appears in the
        partial_assignment, with a sign matching the literal.
    """
    variable, val = lit_to_var_val(literal)
    return variable in partial_assignment and partial_assignment[variable] == val


def clause_is_satisfied(clause, partial_assignment):
    """Checks whether the clause is satisfied by the assignment.

    Args:
      clause: A list of nonzero ints.
      partial_assignment: A dict of variables (ints) to values (bools).

    Returns:
      is_satisfied: True if some literal in the clause is satisfied.
    """
    for literal in clause:
        if literal_is_satisfied(literal, partial_assignment):
            return True
    return False


def find_pure_variable(cnf_formula, variables, partial_assignment):
    """Helper for DPLL.

    A variable is pure if it has the same sign in all unsatisfied clauses
    and if it is not already assigned.

    If a pure variable exists, this function returns the variable and value
    corresponding to the literal. (If multiple exist, return an arbitrary one.)

    If no pure variables exist, return (None, None).

    Args:
      cnf_formula: A list of lists of nonzero integers representing a CNF formula,
        with sign indicating whether the variable is negated.
      variables: A list of positive integers.
      partial_assignment: A dict mapping positive integers to bools, or None for
        an empty assignment.

    Returns:
      variable : A positive integer or None.
      value: A bool or None.
    """
    candidate_to_possible_values  = {v : {True, False} for v in variables \
                                    if v not in partial_assignment}
    for clause in cnf_formula:
        if clause_is_satisfied(clause, partial_assignment):
            continue
        for literal in clause:
            variable, value = lit_to_var_val(literal)
            if variable in candidate_to_possible_values:
                candidate_to_possible_values[variable].discard(not value)
    for candidate, possible_values in candidate_to_possible_values.items():
        if possible_values:
            value = next(iter(possible_values))
            return candidate, value
    return None, None


def find_unit_clause(cnf_formula, partial_assignment):
    """Helper for DPLL.

    A clause is a unit clause if all literals but one are already assigned to
    False. If a unit clause exists, this function returns the variable and value
    corresponding to the literal. (If multiple exist, return an arbitrary one.)

    Args:
      cnf_formula: A list of lists of nonzero integers representing a CNF formula,
        with sign indicating whether the variable is negated.
      variables: A list of positive integers.
      partial_assignment: A dict mapping positive integers to bools, or None for
        an empty assignment.

    Returns:
      variable : A positive integer or None.
      value: A bool or None.
    """
    for clause in cnf_formula:
        unassigned_literal = None
        is_unit_clause = True
        for literal in clause:
            # If the literal is true in the assignment, this is not a unit clause
            if literal_is_satisfied(literal, partial_assignment):
                is_unit_clause = False
                break
            # If the literal is false in the assignment, this could be a unit clause
            elif literal in partial_assignment:
                continue
            # If there is already an unassigned literal, this is not a unit clause
            elif not (unassigned_literal is None):
                is_unit_clause = False
                break
            else:
                unassigned_literal = literal
        if is_unit_clause and not (unassigned_literal is None):
            return lit_to_var_val(unassigned_literal)
    return None, None

### [OPTIONAL] Question

**This question is optional.** Uncomment the tests below if you'd like to work on it. You can find an explanation of it and pseudocode in AIMA section 7.6.

Use your helper functions to complete an implementation of DPLL.

For reference, our solution is **59** line(s) of code.

In [ ]:
def run_inference_dpll(cnf_formula):
    """Find a satisfying assignment for a propositional CNF formula with DPLL.

    Args:
      cnf_formula: A list of lists of nonzero integers representing a CNF formula,
        with sign indicating whether the variable is negated.

    Returns:
      satisfiable: A bool indicating whether some satisfying assignment exists.
      assignment: A dict mapping positive integers to bools, or None if no
        satisfying assignment exists.

    Examples:
      >> run_inference_dpll([[1, -2], [-1, -2]])
      >> (True, {1: True, 2: False}))

      >> run_inference_dpll([[1], [-1]])
      >> (False, None)
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:
""" Uncomment to work on this problem!
assert run_inference_dpll([]) == (True, {})


assert run_inference_dpll([[1]]) == (True, {1: True})


assert run_inference_dpll([[-1]]) == (True, {1: False})


assert run_inference_dpll([[1, 2]]) in [(True, {1: True, 2: True}), (True, {1: True, 2: False}), (True, {1: False, 2: True})]


assert run_inference_dpll([[-1, 2]]) in [(True, {1: True, 2: True}), (True, {1: False, 2: True}), (True, {1: False, 2: False})]


assert run_inference_dpll([[1], [-1]]) == (False, None)


assert run_inference_dpll([[1, 2, 3], [-1, -2, -3], [1, -2, 3], [-1], [-3]]) == (False, None)


assert run_inference_dpll([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17], [18], [19], [20], [21], [22], [23], [24], [25], [26], [27], [28], [29], [30], [31], [32]]) == (True, {1: True, 2: True, 3: True, 4: True, 5: True, 6: True, 7: True, 8: True, 9: True, 10: True, 11: True, 12: True, 13: True, 14: True, 15: True, 16: True, 17: True, 18: True, 19: True, 20: True, 21: True, 22: True, 23: True, 24: True, 25: True, 26: True, 27: True, 28: True, 29: True, 30: True, 31: True, 32: True})
print('Tests passed.')
"""

## Sympy Warmup 1


### Question
Use sympy to determine whether the following formula is satisfiable:
$(\neg x_1 \land x_2) \Rightarrow ((x_2 \lor x_3) \land (x_1 \lor \neg x_3))$.

Note that the return type should be **bool**.


For reference, our solution is **4** line(s) of code.

In [ ]:
def formula1_is_satisfiable():
    """Determines whether the above formula is satisfiable.

    Returns:
      is_satisfiable: A bool indicating whether the formula is satisfiable.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:

assert formula1_is_satisfiable() == True
print('Tests passed.')

## Sympy Warmup 2


### Question
Use sympy to determine whether the following formula is satisfiable:
$(x_1 \lor x_2) \land (\neg x_1 \lor \neg x_2) \land (x_1 \lor \neg x_2) \land (\neg x_1 \lor x_2)$.


For reference, our solution is **4** line(s) of code.

In [ ]:
def formula2_is_satisfiable():
    """Determines whether the above formula is satisfiable.

    Returns:
      is_satisfiable: A bool indicating whether the formula is satisfiable.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:

assert formula2_is_satisfiable() == False
print('Tests passed.')

## Search and Rescue Inference

### Question
Write a program that takes a grid as input and infers unknown values.

Your program should output a new grid with all determinable unknown values replaced with the inferred value. If an unknown value cannot be determined, it should be left unknown.

**Your program should use sympy.**


For reference, our solution is **57** line(s) of code.

In [ ]:
def infer_unknown_values(grid):
    """Fill in any unknown values in the grid that can be inferred.

    Args:
      grid: A list of lists of "F", "U", "S", or "C".

    Returns:
      inferred_grid: A copy of grid with some unknown values replaced.

    Example:
      >> grid = [
      >>   ["F", "U", "C"],
      >>   ["S", "C", "U"],
      >>   ["U", "U", "C"]
      >> ]
      >> infer_unknown_values(grid)
      >> [["F" "S" "C"]
      >>  ["S" "C" "C"]
      >>  ["U" "U" "C"]]
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:

assert infer_unknown_values([["U", "F"]]) == [["S", "F"]]


assert infer_unknown_values([["F", "U", "C"], ["S", "C", "U"], ["U", "U", "C"]]) == [["F", "S", "C"], ["S", "C", "C"], ["U", "U", "C"]]


assert infer_unknown_values([["U", "C", "C"], ["S", "C", "U"], ["U", "U", "C"]]) == [["C", "C", "C"], ["S", "C", "C"], ["F", "S", "C"]]


assert infer_unknown_values([["U", "S", "C", "U"], ["U", "U", "C", "U"], ["U", "S", "C", "U"]]) == [["F", "S", "C", "C"], ["S", "C", "C", "C"], ["F", "S", "C", "C"]]


assert infer_unknown_values([["U", "U", "C", "U", "U", "U", "U", "U"], ["C", "U", "U", "U", "U", "U", "U", "U"], ["U", "U", "U", "U", "U", "U", "U", "U"], ["U", "U", "U", "U", "U", "U", "C", "C"], ["U", "U", "U", "U", "U", "U", "C", "C"], ["U", "C", "U", "U", "U", "U", "U", "U"], ["U", "U", "U", "F", "U", "U", "U", "U"], ["U", "U", "U", "U", "U", "U", "U", "U"]]) == [["C", "C", "C", "U", "U", "U", "U", "U"], ["C", "U", "U", "U", "U", "U", "U", "U"], ["U", "U", "U", "U", "U", "U", "U", "U"], ["U", "U", "U", "U", "U", "U", "C", "C"], ["U", "U", "U", "U", "U", "U", "C", "C"], ["U", "C", "U", "S", "U", "U", "U", "U"], ["U", "U", "S", "F", "S", "U", "U", "U"], ["U", "U", "U", "S", "U", "U", "U", "U"]]
print('Tests passed.')

# First-Order Logic

## Imports and Utilities
**Note**: these imports and functions are available in catsoop. You do not need to copy them in.

In [ ]:
from collections import namedtuple
from itertools import chain

# Take a moment to review the namedtuple documentation:
# https://docs.python.org/3/library/collections.html#collections.namedtuple

## Common logic data structures
Not = namedtuple("Not", ["sentence"])
And = namedtuple("And", ["sentence1", "sentence2"])
Or = namedtuple("Or", ["sentence1", "sentence2"])
Implies = namedtuple("Implies", ["sentence1", "sentence2"])

## Propositional logic data structures
Proposition = str # Name of the proposition
PropositionalModel = dict # Proposition -> bool

## Example of PropositionalModel, used in tests
IS_RAINING = Proposition("is-raining")
IS_SUNNY = Proposition("is-sunny")
NEED_UMBRELLA = Proposition("need-umbrella")
PROP_MODEL = PropositionalModel({
  IS_RAINING: True,
  IS_SUNNY: False,
  NEED_UMBRELLA: True,
})

## First-order logic data structures
Object = str
Constant = namedtuple("Constant", ["name"])
Variable = namedtuple("Variable", ["name"])
Predicate = namedtuple("Predicate", ["name", "arity"])
Atom = namedtuple("Atom", ["predicate", "terms"])
ForAll = namedtuple("ForAll", ["variable", "sentence"])
Exists = namedtuple("Exists", ["variable", "sentence"])
Interpretation = namedtuple("Interpretation", ["constant_map", "predicate_map"])
FOLModel = namedtuple("FOLModel", ["objects", "interpretation"])

# Example of FOLModel, used in tests
TOM = Object("tom-object")
NOMSY = Object("nomsy-object")
PUDDLES = Object("puddles-object")
OBJECTS = {TOM, NOMSY, PUDDLES}
CT, CN, CP = Constant("Tom"), Constant("Nomsy"), Constant("Puddles")
X, Y = Variable("X"), Variable("Y")
Likes = Predicate("Likes", 2)
IsDog = Predicate("IsDog", 1)
CONSTANT_MAP = {CT : TOM, CN: NOMSY, CP: PUDDLES}
PREDICATE_MAP = {
  Likes: {(TOM, NOMSY), (TOM, PUDDLES), (NOMSY, TOM)},
  IsDog: {(NOMSY,), (PUDDLES,)},
}
INTERPRETATION = Interpretation(constant_map=CONSTANT_MAP,
                                predicate_map=PREDICATE_MAP)
FOL_MODEL = FOLModel(objects=OBJECTS, interpretation=INTERPRETATION)

## First-order CNF logical data structures
Literal = namedtuple("Literal", ["atom", "is_positive"])
def negate(literal):
  return Literal(literal.atom, not literal.is_positive)
def Clause(literals=tuple()):
  return frozenset(literals)
def CNFSentence(clauses):
  return set(clauses)

def unify_var(v, t, th):
  """Unify a variable and a (non-function) term.

  Args:
    v: A Variable.
    t: A Constant or Variable.
    th: A dict mapping variables to objects.

  Returns:
    theta: A dict mapping variables to objects
      or None if no substitution exists.
  """
  if th is None: return None
  if v == t:
    return th
  elif v in th:
    return unify(th[v], t, th)
  elif t in th:
    return unify_var(v, th[t], th)
  else:
    return compose_subst(th, {v:t})

def compose_subst(s1, s2):
  return dict(chain(((x, subst(t, s2)) for (x, t) in s1.items() if x != subst(t, s2)),
                    ((x, t) for (x, t) in s2.items() if not x in s1)))


def subst(a, th):
  """Substitute variables for a single Literal.

  Args:
    a: A Constant, Variable, Atom, Literal, list, tuple, or frozenset.
    th: A dict mapping variables to objects.

  Returns:
    b: Same type as a.
  """
  if isinstance(a, Constant):
    return a
  elif isinstance(a, Variable):
    return th[a] if a in th else a
  elif isinstance(a, Atom):
    return Atom(a.predicate, subst(a.terms, th))
  elif isinstance(a, Literal):
    return Literal(subst(a.atom, th), a.is_positive)
  elif isinstance(a, (list, tuple)):
    return tuple(subst(x, th) for x in a)
  elif isinstance(a, frozenset):
    return frozenset(subst(x, th) for x in a)
  else:
    raise Exception('Unknown type:'+str(a))


def unify(a, b, th):
  """Run unification.

  Args:
    a: A Constant, Variable, Atom, Literal, or list or tuple of terms.
    b: A Constant, Variable, Atom, Literal, or list or tuple of terms.
    th: A dict mapping variables to objects, or None if no unifier exists.

  Returns:
    new_th: A new dict representing a unifier, or None if none exist.

  Examples:
    # Example 1: Unifying a literal and its negation
    x = Literal(Atom("P", [Variable("X")]), True)
    y = Literal(Atom("P", [Constant("a")]), False)
    unify(x, y, {})  # Returns: None

    # Example 2: Unifying two literals with matching predicates
    x = Literal(Atom("P", [Variable("X")]), True)
    y = Literal(Atom("P", [Constant("a")]), True)
    unify(x, y, {})  # Returns: {'X': Constant("a")}

    # Example 3: Unifying two atoms with matching predicates
    atom1 = Atom("parent", [Variable("X"), Constant("John")])
    atom2 = Atom("parent", [Constant("Mary"), Constant("John")])
    unify(atom1, atom2, {})  # Returns: {'X': Constant("Mary")}

    # Example 4: Unifying a variable with a constant
    x = Variable("x")
    y = Constant("Y")
    unify(x, y, {})  # Returns: {'x': Constant("Y")}

    # Example 5: Unifying two constants (successful)
    a = Constant("A")
    b = Constant("A")
    unify(a, b, {})  # Returns: {}

    # Example 6: Unifying two different constants (fails)
    a = Constant("A")
    b = Constant("B")
    unify(a, b, {})  # Returns: None

    # Example 7: Unifying nested structures
    term1 = Atom("likes", [Variable("X"), Atom("food", [Constant("pizza")])])
    term2 = Atom("likes", [Constant("Alice"), Atom("food", [Constant("pizza")])])
    unify(term1, term2, {})  # Returns: {'X': Constant("Alice")}
  """
  if th is None: return None
  if isinstance(a, Constant):
    if isinstance(b, Variable):
      return unify_var(b, a, th)
    else:
      return th if a == b else None
  elif isinstance(a, Variable):
    return unify_var(a, b, th)
  elif isinstance(b, Variable):
    return unify_var(b, a, th)
  elif isinstance(a, Atom):
    return unify(a.terms, b.terms, th) if (isinstance(b, Atom) and a.predicate == b.predicate) else None
  elif isinstance(a, Literal):
    return unify(a.atom, b.atom, th) if (isinstance(b, Literal) and a.is_positive == b.is_positive) else None
  elif isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
    if len(a) == 0 or len(b) == 0:
      return th if len(a) == len(b) else None
    else:
      return unify(a[0], b[0], unify(a[1:], b[1:], th))
  else:
    raise Exception('Unknown type:'+str(a))



## Atom Evaluation


### Question
*Note: for these questions, refer to the top of the Colab notebook.*
Write a function that takes a FOL atom and evaluates it against a single model.

For reference, our solution is **10** line(s) of code.

In [ ]:
def evaluate_atom(atom, model, substitution):
    """Evaluate if an atom holds under the model.

    Args:
      atom: An Atom.
      model: A FOLModel.
      substitution: A dict mapping variables to objects.

    Returns:
      holds: A bool.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:
def evaluate_atom_test1():
  assert evaluate_atom(Atom(IsDog, (CN,)), FOL_MODEL, {}) == True

evaluate_atom_test1()


def evaluate_atom_test2():
  assert evaluate_atom(Atom(IsDog, (X,)), FOL_MODEL, {X: NOMSY}) == True

evaluate_atom_test2()


def evaluate_atom_test3():
  assert evaluate_atom(Atom(IsDog, (CT,)), FOL_MODEL, {}) == False

evaluate_atom_test3()


def evaluate_atom_test4():
  assert evaluate_atom(Atom(IsDog, (X,)), FOL_MODEL, {X: TOM}) == False

evaluate_atom_test4()


def evaluate_atom_test5():
  assert evaluate_atom(Atom(Likes, (X, Y)), FOL_MODEL, {X: TOM, Y: NOMSY}) == True

evaluate_atom_test5()

print('Tests passed.')

## First-order Logic Sentence Evaluation


### Question
Use your implementation of evaluate_atom to complete the following implementation of FOL sentence evaluation.

For reference, our solution is **37** line(s) of code.

In addition to all the utilities defined at the top of the Colab notebook, the following functions are available in this question environment: `evaluate_atom`. You may not need to use all of them.

In [ ]:
def evaluate_fol_sentence(sentence, model, substitution=None):
    """Evaluate a first-order logic sentence against a single model.

    Note that Literals are not used here (we use them in later problems).

    Be careful about updating `substitution` recursively. You may want
    to create a copy of the dict (`substitution.copy()`) before each
    recursive call.

    Args:
      sentence: An Atom, And, Or, Not, Implies, ForAll, or Exists.
      model: A FOLModel.
      substitution: A dict mapping variables to objects, or None,
        representing an empty dict.

    Returns:
      holds: A bool representing the truth value of the sentence
        under the model.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:
def evaluate_fol_sentence_test1():
  assert evaluate_fol_sentence(And(Atom(IsDog, (CN,)), Atom(IsDog, (CP,))), FOL_MODEL) == True

evaluate_fol_sentence_test1()


def evaluate_fol_sentence_test2():
  assert evaluate_fol_sentence(Or(Atom(IsDog, (CT,)), Not(Atom(IsDog, (CP,)))), FOL_MODEL) == False

evaluate_fol_sentence_test2()


def evaluate_fol_sentence_test3():
  assert evaluate_fol_sentence(And(Atom(IsDog, (CN,)), Atom(IsDog, (CP,))), FOL_MODEL) == True

evaluate_fol_sentence_test3()


def evaluate_fol_sentence_test4():
  assert evaluate_fol_sentence(Exists(X, And(Atom(IsDog, (X,)), Not(Atom(Likes, (CT, X))))), FOL_MODEL) == False

evaluate_fol_sentence_test4()


def evaluate_fol_sentence_test5():
  assert evaluate_fol_sentence(Exists(X, Exists(Y, Atom(Likes, (Y, X)))), FOL_MODEL) == True

evaluate_fol_sentence_test5()

print('Tests passed.')

## FOL Binary Resolution


### Question
Complete the following implementation of first-order binary resolution.
Given two clauses, return all possible new clauses that result from applying the binary resolution rule.
As part of this you'll want to use the `unify` and `subst` helper functions defined above. Take a look at the docstring - specifically the examples listed there - to get an idea of what precisely the `unify` function does. Recall that as part of applying resolution, you'll potentially need to (1) get rid of certain literals, and (2) make substitutions into parts of each input `Clause`.

<b>Hint 1</b>: The helper function `unify` may return an empty dict for certain inputs. Note that this means the inputs
can be successfully unified without any variable substitution.

<b>Hint 2</b>: In your code, you can use `if unify(a, b, {}) is not None` to test if the unification is successful.

<b>Hint 3</b>: Look at the examples of using the `unify` function in the docstring. In particular - pay close attention to what happens when you unify a literal with its negation? What does this mean for how you should be calling this function on specific `Literal`s from your two input `Clause`s?


For reference, our solution is **8** line(s) of code.

In [ ]:
def binary_resolution(clause1, clause2):
    """Return all new clauses resulting from binary resolution.

    Args:
      clause1: A Clause (frozenset of Literals).
      clause2: A Clause (frozenset of Literals).

    Returns:
      clauses: A set of new Clauses.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:
clause1 = Clause([Literal(Atom(IsDog, (CT,)), True)])
clause2 = Clause([Literal(Atom(IsDog, (CT,)), False)])
assert binary_resolution(clause1, clause2) == {Clause()}


clause1 = Clause([Literal(Atom(IsDog, (X,)), True)])
clause2 = Clause([Literal(Atom(IsDog, (Y,)), False)])
assert binary_resolution(clause1, clause2) == {Clause()}


clause1 = Clause([Literal(Atom(IsDog, (X,)), True)])
clause2 = Clause([Literal(Atom(IsDog, (Y,)), True)])
assert binary_resolution(clause1, clause2) == set()


# All dogs are liked by Tom
clause1 = Clause([Literal(Atom(IsDog, (X,)), False), Literal(Atom(Likes, (CT, X)), True)])
# Nomsy is a dog
clause2 = Clause([Literal(Atom(IsDog, (CN,)), True)])
# So Nomsy must be liked by Tom
assert binary_resolution(clause1, clause2) == {Clause([Literal(Atom(Likes, (CT, CN)), True)])}

print('Tests passed.')

## FOL Resolution Prover


### Question
Complete the following implementation of a first-order resolution prover. Given a sentence in CNF form, and a single query clause, check if the sentence entails the query. See unit tests for examples.

For reference, our solution is **17** line(s) of code.

In addition to all the utilities defined at the top of the Colab notebook, the following functions are available in this question environment: `binary_resolution`. You may not need to use all of them.

In [ ]:
def resolution_prover(kb, q):
    """Check if a knowledge base entails a query.

    That is, if kb ^ not q entails False.
    That is, if we can prove False from kb ^ not q.
    May run forever.

    Args:
      kb: A CNFSentence.
      q: A single Clause.

    Returns:
      entails: True if kb entails q.
    """
    raise NotImplementedError("Implement me!")

### Tests

In [ ]:
# All dogs are liked by Tom
kb_clause1 = Clause([Literal(Atom(IsDog, (X,)), False), Literal(Atom(Likes, (CT, X)), True)])
# Nomsy is a dog
kb_clause2 = Clause([Literal(Atom(IsDog, (CN,)), True)])
kb = CNFSentence([kb_clause1, kb_clause2])
# Tom likes Nomsy
query = Clause([Literal(Atom(Likes, (CT, CN)), True)])
assert resolution_prover(kb, query) == True


# Tom does not like Nomsy (impossible!)
query = Clause([Literal(Atom(Likes, (CT, CN)), False)])
assert resolution_prover(kb, query) == False


# Russell & Norvig example
American = Predicate("American", 1)
Weapon = Predicate("Weapon", 1)
Sells = Predicate("Sells", 3)
Hostile = Predicate("Hostile", 1)
Criminal = Predicate("Criminal", 1)
Missile = Predicate("Missile", 1)
Enemy = Predicate("Enemy", 1)
Owns = Predicate("Owns", 2)
Nono = Constant("Nono")
America = Constant("America")
West = Constant("West")
M1 = Constant("M1")

clause1 = Clause([
  Literal(Atom(American, (Variable("x1"),)), False),
  Literal(Atom(Weapon, (Variable("y1"),)), False),
  Literal(Atom(Sells, (Variable("x1"),Variable("y1"),Variable("z1"))), False),
  Literal(Atom(Hostile, (Variable("z1"),)), False),
  Literal(Atom(Criminal, (Variable("x1"),)), True),
])
clause2 = Clause([
  Literal(Atom(Missile, (Variable("x2"),)), False),
  Literal(Atom(Owns, (Nono, Variable("x2"),)), False),
  Literal(Atom(Sells, (West, Variable("x2"), Nono)), True),
])
clause3 = Clause([
  Literal(Atom(Enemy, (Variable("x3"), America)), False),
  Literal(Atom(Hostile, (Variable("x3"),)), True),
])
clause4 = Clause([
  Literal(Atom(Missile, (Variable("x4"),)), False),
  Literal(Atom(Weapon, (Variable("x4"),)), True),
])
clause5 = Clause([
  Literal(Atom(Owns, (Nono, M1)), True),
])
clause6 = Clause([
  Literal(Atom(Missile, (M1,)), True),
])
clause7 = Clause([
  Literal(Atom(American, (West,)), True),
])
clause8 = Clause([
  Literal(Atom(Enemy, (Nono, America)), True),
])
kb = CNFSentence([clause1, clause2, clause3, clause4, clause5, clause6, clause7, clause8])
query = Clause([
  Literal(Atom(Criminal, (West,)), True),
])
assert resolution_prover(kb, query) == True

print('Tests passed.')